In [1]:
import numpy 
import torch 
import torch.nn as nn 
from torch.utils.data import TensorDataset, DataLoader

In [8]:
class LSTM(nn.Module):

    def __init__(self,input_size,hidden_size):
        super().__init__()
        self.rnn = nn.LSTM(input_size=input_size,hidden_size=hidden_size,batch_first=True)
        self.fc = nn.Linear(hidden_size,2)

    def forward(self,X):

        output,(hn,cn) = self.rnn(X)
        last_hidden = hn[-1]
        logits = self.fc(last_hidden)
        return logits

In [9]:
def create_dataset(num_samples, seq_length=10, input_size=5):
    X = torch.randn(num_samples, seq_length, input_size)
    y = (X.sum(dim=(1, 2)) > 0).long()

    return X, y


X_train, y_train = create_dataset(1000)
X_test, y_test = create_dataset(200)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True
)

test_loader = DataLoader(
    test_dataset, batch_size=32
)

In [10]:
model = LSTM(input_size=5, hidden_size=16)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for e in range(20):

    model.train()

    total_loss = 0
    correct = 0 
    total = 0 

    for X,y in train_loader:

        logits = model(X)

        loss = criterion(logits,y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)


        predictions = logits.argmax(dim=1)
        correct += (predictions == y).sum().item()
        total += y.size(0)

    train_loss = total_loss/total
    train_accuracy = correct/total
    print(
        f"Epoch [{e + 1:02d}/{e}] "
        f"Loss: {train_loss:.4f} "
        f"Accuracy: {train_accuracy:.4f}"
    )

    model.eval()

    correct = 0 
    total = 0 
    test_loss = 0 

    with torch.no_grad():

        for Xt,yt in test_loader:
            logits = model(Xt)

            loss = criterion(logits,yt)

            test_loss+=loss.item() * Xt.size(0)

            predictions = logits.argmax(dim=1)

            correct += (predictions == yt).sum().item()
            total+= yt.size(0)

        test_accuracy = correct/total 
        total_loss = test_loss/total
    print("\nTest Results")
    print(f"Test Loss: {total_loss:.4f}")
    print(f"Test Accuracy: {test_accuracy:.4f}")



Epoch [01/0] Loss: 0.6932 Accuracy: 0.5170

Test Results
Test Loss: 0.6842
Test Accuracy: 0.6000
Epoch [02/1] Loss: 0.6692 Accuracy: 0.7080

Test Results
Test Loss: 0.6578
Test Accuracy: 0.7500
Epoch [03/2] Loss: 0.6136 Accuracy: 0.7880

Test Results
Test Loss: 0.5515
Test Accuracy: 0.8450
Epoch [04/3] Loss: 0.4024 Accuracy: 0.9160

Test Results
Test Loss: 0.3120
Test Accuracy: 0.8600
Epoch [05/4] Loss: 0.2702 Accuracy: 0.8920

Test Results
Test Loss: 0.2452
Test Accuracy: 0.9100
Epoch [06/5] Loss: 0.2103 Accuracy: 0.9370

Test Results
Test Loss: 0.2058
Test Accuracy: 0.9100
Epoch [07/6] Loss: 0.1749 Accuracy: 0.9470

Test Results
Test Loss: 0.1767
Test Accuracy: 0.9350
Epoch [08/7] Loss: 0.1509 Accuracy: 0.9510

Test Results
Test Loss: 0.1566
Test Accuracy: 0.9600
Epoch [09/8] Loss: 0.1371 Accuracy: 0.9510

Test Results
Test Loss: 0.1382
Test Accuracy: 0.9550
Epoch [10/9] Loss: 0.1249 Accuracy: 0.9530

Test Results
Test Loss: 0.1390
Test Accuracy: 0.9550
Epoch [11/10] Loss: 0.1174 Acc